<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [ ]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print('exact duplicate rows:', df.duplicated().sum())

(8, 6)
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64
exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1) Two exact duplicate rows (order_id 1 appears twice, byte-for-byte identical).
2) price is stored as text with inconsistent formatting — some values have $ and two decimals (7.50), others are bare numbers (7.5, 12, 6).
3) qty has a literal string 'NULL', a missing/blank value, and a negative value (likely a refund).
4) category has the same category spelled multiple ways (Food/food, RainGear/rain-gear), plus a T-shirt classified under a one-off category (Apparel) that doesn't appear anywhere else.
5) item has inconsistent casing (Cheeseburger/cheese burger), trailing whitespace ('UVA T-Shirt '), and one row with no item name at all.
6) ts is stored in at least three different formats (ISO with T, ISO with space, US MM/DD/YYYY) and is missing entirely for one row.


### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
clean['price'] = (
    clean['price'].astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
    .astype(float)
)

assert clean['price'].dtype == float
log('price', 'price arrived as text ($7.50, 7.5, etc.) — stripped $ signs and whitespace, converted to float', len(clean))

[price] price arrived as text ($7.50, 7.5, etc.) — stripped $ signs and whitespace, converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# Decision 1: missing quantity — fill with 1 rather than drop the order.
# We still know the item, category, and price for that row, so dropping it
# would throw away real information just because one field was blank.
clean['qty'] = clean['qty'].fillna(1)
log('qty missing', f'filled {missing} missing quantity value(s) with 1 (assumed single unit)', missing)

# Decision 2: negative quantity (refund) — keep it as-is. A negative qty
# represents real reversed revenue; zeroing or dropping it would overstate
# what the vendor actually earned that day.
log('qty negative', f'kept {negative} negative quantity value(s) as refunds', negative)

[qty missing] filled 1 missing quantity value(s) with 1 (assumed single unit) (1 row(s))
[qty negative] kept 1 negative quantity value(s) as refunds (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:
print('before:', sorted(clean['category'].unique()))

clean['category'] = (
    clean['category'].str.lower().str.strip().str.replace('-', '', regex=False)
)

CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',   # judgment call: one lone T-shirt row doesn't earn its own category
    'raingear': 'RainGear',
}
clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))
log('category', f'normalized casing/punctuation, merged Apparel into Merch — {clean["category"].nunique()} categories remain', len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Food', 'Merch', 'RainGear']
[category] normalized casing/punctuation, merged Apparel into Merch — 3 categories remain (7 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
print('before:', sorted(clean['item'].fillna('<missing>').unique()))

clean['item'] = clean['item'].str.strip().str.lower()

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva t-shirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho',
}
clean['item'] = clean['item'].map(ITEM_MAP)

missing_item = clean['item'].isna().sum()
clean['item'] = clean['item'].fillna('Unknown item')

print('after: ', sorted(clean['item'].unique()))
log('item', f'normalized spelling/casing/whitespace, labeled {missing_item} row(s) with no item as "Unknown item" instead of dropping', len(clean))

before: ['<missing>', 'Cheeseburger', 'Foam Finger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Foam Finger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown item']
[item] normalized spelling/casing/whitespace, labeled 1 row(s) with no item as "Unknown item" instead of dropping (7 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [9]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')

failed = clean['ts'].isna().sum()
print(f'{failed} timestamp(s) missing/unparseable')

clean['hour'] = clean['ts'].dt.hour

log('timestamps', f'parsed ts to datetime (mixed formats: ISO with T, ISO with space, MM/DD/YYYY), coerced {failed} to NaT, added hour column', len(clean))


4 timestamp(s) missing/unparseable
[timestamps] parsed ts to datetime (mixed formats: ISO with T, ISO with space, MM/DD/YYYY), coerced 4 to NaT, added hour column (7 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [14]:
assert len(clean) == 7, 'expected 7 rows after dropping the exact duplicate'
assert clean['price'].dtype == float
assert clean['qty'].isna().sum() == 0, 'qty should have no missing values after cleaning'
assert clean['category'].nunique() == 3, 'categories should collapse to Food, Merch, RainGear'
assert clean['item'].isna().sum() == 0, 'item should have no missing values after cleaning'
assert clean.duplicated().sum() == 0, 'no exact duplicates should remain'
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), 'ts should be a real datetime'

clean['revenue'] = clean['qty'] * clean['price']

print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', round(clean['revenue'].sum(), 2))
print('distinct categories:', clean['category'].nunique())

rows: 7
units: 8.0
revenue: 100.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [11]:
show_log()

,step,decision,rows
0,price,"price arrived as text ($7.50, 7.5, etc.) — str...",7
1,qty missing,filled 1 missing quantity value(s) with 1 (ass...,1
2,qty negative,kept 1 negative quantity value(s) as refunds,1
3,category,"normalized casing/punctuation, merged Apparel ...",7
4,item,"normalized spelling/casing/whitespace, labeled...",7
5,timestamps,parsed ts to datetime (mixed formats: ISO with...,7
6,timestamps,parsed ts to datetime (mixed formats: ISO with...,7


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [15]:
# Checkpoint
rows_after = len(clean)
revenue_after = round(clean['revenue'].sum(), 2)
biggest_decision = 'Kept the negative-quantity refund (order 5) instead of dropping or zeroing it, since it represents real reversed revenue'
revenue_other_way = round(revenue_after + 18, 2)

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 7
revenue: 100.5
decision that mattered: Kept the negative-quantity refund (order 5) instead of dropping or zeroing it, since it represents real reversed revenue
revenue the other way: 118.5
